# Wikipedia Agent demo

A [Pydantic AI](https://ai.pydantic.dev/) agent that answers questions using **only English Wikipedia**. For each question it:

1. calls `search` to find the exact page title,
2. calls `get_page` to read that page (long pages come in chunks),
3. returns a structured `WikiAnswer` (`answer`, `sources`, `found`) that cites only the pages it actually read.

This notebook walks through the tools on their own, a full agent run, how to see which tools the agent called, follow-up questions, other languages, unanswerable questions and usage limits.

**Before running:** `uv sync`, and a `.env` file in the project root with `OPENAI_API_KEY` (optionally `WIKI_AGENT_MODEL`). Select the project's `.venv` as the kernel.

## Setup

In [1]:
import os

os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")

from collections.abc import Sequence

from pydantic_ai import RunUsage
from pydantic_ai.exceptions import UsageLimitExceeded
from pydantic_ai.messages import ModelMessage
from pydantic_ai.usage import UsageLimits

from wiki_agent import WikiAnswer, WikipediaTools, create_agent, print_messages

wiki = WikipediaTools()
agent = create_agent(tools=wiki)
print("Model:", agent.model)

Model: openai:gpt-4o-mini


Two small helpers used below: `show` prints an answer with its sources, and `tools_used` lists the tool calls in a run's messages. Pydantic AI returns the structured answer through a special `final_result` tool call, so `tools_used` leaves that out.

In [2]:
def show(answer: WikiAnswer) -> None:
    """Print the answer, whether it was found, and the pages it cites."""
    print(answer.answer)
    print(f"\nfound={answer.found}")
    for source in answer.sources:
        print(f"- {source.title}: {source.url}")


def tools_used(messages: Sequence[ModelMessage]) -> list[str]:
    """Names of the Wikipedia tools called in `messages`, in order (skips the `final_result` output call)."""
    return [
        part.tool_name
        for message in messages
        for part in message.parts
        if part.part_kind == "tool-call" and part.tool_name != "final_result"
    ]

## 1. The tools on their own (no LLM)

The agent's tools are plain methods on `WikipediaTools`, so you can call them directly to see exactly what the model sees. `search` returns titles and snippets:

In [3]:
wiki.search("capybara")

[{'title': 'Capybara',
  'snippet': 'The capybara (/kæpɪˈbɑːrə/ kap-ih-BAR-uh) or greater capybara (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South'},
 {'title': 'Lesser capybara',
  'snippet': 'The lesser capybara (Hydrochoerus isthmius) is a large semi-aquatic rodent found in South America that has vast similarities with, yet subtle differences'},
 {'title': 'Capybara (disambiguation)',
  'snippet': 'Look up capybara\xa0or Capybara in Wiktionary, the free dictionary. The capybara is a giant cavy rodent native to South America. Capybara may also refer to:'},
 {'title': 'Capybara Games',
  'snippet': 'Capybara Games is an independent game studio based in Toronto, Ontario, Canada. The studio was founded in 2003 from a collection of Toronto IGDA members'},
 {'title': 'Capybara Party',
  'snippet': 'The Capybara Party (Italian: Partito Capibara) is an Italian political movement founded online in September 2022. Characterized as a post-work and ant

`get_page` returns cleaned wikitext. Pages longer than `chunk_size` (8000 characters) end with a `[truncated: ...]` note that tells the model which `offset` to read next:

In [4]:
page = wiki.get_page("Capybara")
print(page[:500])
print("\n...\n")
print(page[-150:])

{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = 
| fossil_range = 
| image = Capybaracropped.jpg
| image_alt = A capybara on a river bank
| image_caption = In [[Encontro das Águas State Park]], Brazil
| genus = Hydrochoerus
| species = hydrochaeris
| authority = ([[Carl Linnaeus|Linnaeus]], [[12th edition of Systema Naturae|1766]])
| range_map = Capybara range.svg
| range_map_alt = Map of South America, showing the species occurs across most of it, except for the western coas

...

o more abundant reeds during the dry season. Plants that capybaras eat during

[truncated: showing 0–8000 of 19364 chars; call again with offset=8000]


## 2. Ask a question

`agent.run` lets the model decide which tools to call. The result's `output` is a `WikiAnswer`.

In [5]:
result = await agent.run("Where do capybaras live?")
show(result.output)

Capybaras are native to all countries in South America except Chile. They inhabit semi-aquatic environments, typically living near bodies of freshwater such as rivers, lakes, ponds, swamps, and wetlands. Their preferred habitats include savannas and dense forests, and they are commonly found in lowland areas, though they can also exist at elevations up to 1,500 meters in some regions. Because of their adaptability, capybaras have also been spotted in various locations worldwide where they have escaped captivity, such as Florida and parts of California.

found=True
- Capybara: https://en.wikipedia.org/wiki/Capybara


## 3. Which tools did the agent use?

`result.new_messages()` holds every step of the run. `print_messages` prints the prompt, each tool call with its arguments, and the size of each tool result. The last call, `final_result`, is the model handing back its `WikiAnswer`:

In [6]:
print_messages(result.new_messages())

USER: Where do capybaras live?
TOOL CALL: search {"query":"capybara habitat"}
TOOL RETURN: search (998 chars)
TOOL CALL: get_page {"title":"Capybara"}
TOOL RETURN: get_page (8073 chars)
TOOL CALL: get_page {"title":"Capybara","offset":8000}
TOOL RETURN: get_page (8078 chars)
TOOL CALL: get_page {"title":"Capybara","offset":16000}
TOOL RETURN: get_page (3364 chars)
TOOL CALL: final_result {"answer":"Capybaras are native to all countries in South America except Chile. They inhabit semi-aquatic environments, typically living near bodies of freshwater such as rivers, lakes, ponds, swamps, and wetlands. Their preferred habitats include savannas and dense forests, and they are commonly found in lowland areas, though they can also exist at elevations up to 1,500 meters in some regions. Because of their adaptability, capybaras have also been spotted in various locations worldwide where they have escaped captivity, such as Florida and parts of California.","sources":[{"title":"Capybara"}],"foun

Or just the tool names, plus the totals from `result.usage`:

In [7]:
print("Tools called:", tools_used(result.new_messages()))
print("Usage:", result.usage)

Tools called: ['search', 'get_page', 'get_page', 'get_page']
Usage: RunUsage(cost=Decimal('0.00237150'), details={'reasoning_tokens': 0}, input_tokens=14938, output_tokens=218, requests=5, tool_calls=4)


## 4. Watch tool calls live

`agent.iter` steps through the run node by node, so you can print each tool call as it happens rather than afterwards. `ask_live` does that and returns the same result object as `agent.run`.

In [8]:
async def ask_live(question: str, message_history: Sequence[ModelMessage] | None = None):
    """Run the agent, printing each tool call as the model makes it."""
    async with agent.iter(question, message_history=message_history) as run:
        async for node in run:
            if agent.is_call_tools_node(node):
                for part in node.model_response.parts:
                    if part.part_kind == "tool-call" and part.tool_name != "final_result":
                        print(f"-> {part.tool_name}({part.args_as_dict()})")
    return run.result


live = await ask_live("When was the Panama Canal opened, and how long is it?")
print()
show(live.output)

-> search({'query': 'Panama Canal opening date and length'})


-> get_page({'title': 'Panama Canal'})
-> get_page({'title': 'History of the Panama Canal'})



The Panama Canal officially opened on August 15, 1914. It is an artificial waterway that is 82 kilometers (51 miles) long.

found=True
- Panama Canal: https://en.wikipedia.org/wiki/Panama_Canal
- History of the Panama Canal: https://en.wikipedia.org/wiki/History_of_the_Panama_Canal


## 5. Follow-up questions

Passing `message_history` lets the agent see the earlier turn, so "they" resolves to capybaras. Use `new_messages()` to see only this turn's tool calls.

In [9]:
followup = await agent.run("And what do they eat?", message_history=result.all_messages())
show(followup.output)
print("\nTools called this turn:", tools_used(followup.new_messages()))

Capybaras are herbivores and primarily graze on grasses and aquatic plants. They are selective feeders, often choosing specific leaves while disregarding others nearby. In addition to grasses, their diet can include fruits and tree bark. During the dry season, capybaras eat a wider variety of plants due to fewer available species. They are also known to be coprophagous, meaning they consume their own feces to help digest cellulose from their food and extract maximum nutrition. This behavior allows them to optimize their nutrient intake, particularly vitamins and proteins.

found=True
- Capybara: https://en.wikipedia.org/wiki/Capybara

Tools called this turn: ['search', 'get_page', 'get_page']


## 6. Other languages

Ask in any language. The agent still searches English Wikipedia, then replies in the language of the question.

In [10]:
spanish = await agent.run("¿Cuál es la montaña más alta de Costa Rica?")
show(spanish.output)

La montaña más alta de Costa Rica es el Cerro Chirripó, con una elevación de 3,821 metros (12,536 pies). Forma parte de la Cordillera de Talamanca y se encuentra en el Parque Nacional Chirripó.

found=True
- Cerro Chirripó: https://en.wikipedia.org/wiki/Cerro_Chirrip%C3%B3


## 7. When Wikipedia doesn't have the answer

The agent answers only from pages it read. If they don't contain the answer, it sets `found=False` instead of guessing.

In [11]:
unknown = await agent.run("What was the capybara's favorite song in 1850?")
show(unknown.output)

The information about a capybara's favorite song in 1850 is not documented or available in historical records or reputable sources. Capybaras, as animals, do not have recognized favorites of human music or songs, especially from a specific year like 1850.

found=False


## 8. Usage and limits

Each run reports its `RunUsage` (requests, tokens, tool calls). Add them up for the whole session:

In [12]:
runs = [result, live, followup, spanish, unknown]
total = sum((run.usage for run in runs), RunUsage())
total

RunUsage(cache_read_tokens=2816, cost=Decimal('0.00888900'), details={'reasoning_tokens': 0}, input_tokens=57516, output_tokens=788, requests=18, tool_calls=14)

Every run is capped by a default budget of 12 model requests, so a model stuck in a loop gets stopped. You can pass a different `usage_limits`. Here a limit of 2 requests stops the run before the agent can search, read a page and answer:

In [13]:
try:
    await agent.run("Summarize the history of the Roman Empire.", usage_limits=UsageLimits(request_limit=2))
except UsageLimitExceeded as error:
    print("Stopped:", error)

Stopped: The next request would exceed the request_limit of 2. Consider raising the limit, or see the docs on usage limits for budget-aware patterns: https://pydantic.dev/docs/ai/core-concepts/agent/#usage-limits
